# Step 3 — Iterative TPS Auto-Matching

Expands seed landmarks (from step_2) into a full match table using iterative
thin-plate spline projection + mutual nearest-neighbour verification.

**Algorithm** (`run_auto_matching`):
1. Fit TPS on accepted landmarks.
2. Project all unmatched CZ cells → HCR pixel space.
3. Forward-match: choose nearest HCR GFP+ cell by spot density among k=5 nearest.
4. Check mutual NN; optionally gate with classifier (if available).
5. Accept passing candidates, add to landmark pool.
6. Subsample landmarks if pool > 500 (spatial grid; keeps TPS well-conditioned).
7. Repeat until convergence (< 0.5% new matches per iteration) or max_iters.

**Inputs** (from `/scratch/{subject}_{date}_coreg_initial/`):
- `*_landmarks_initial.csv` — seed landmarks from step_2
- `*_rigid_initial.json` — rotation + translation (for reference)

**Outputs** (in `/scratch/{subject}_{date}_coreg_initial/`):
- `*_matches.csv` — all accepted (czstack_cell_id, hcr_cell_id, iter_matched, match_probability)
- `fig_matching_summary.png` — per-iteration match counts + distance histogram

In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
SUBJECT_ID   = "790322"          # e.g. "790322", "788406", ...
CZSTACK_DATE = "2025-07-10"

# GFP+ threshold (should match step_2)
GFP_COUNT_THRESHOLD = 5

# Iterative matching parameters
MAX_ITERATIONS          = 10
K_NEIGHBORS             = 5      # candidates per CZ cell in forward match
HCR_VALUE_FEATURE       = "density"  # feature for choose_max_count_nearest_neighbor
DISTANCE_THRESHOLD_UM   = 30.0   # accept radius when no classifier (MNN-only mode)
ACCEPT_PROB_THRESHOLD   = 0.6    # classifier probability gate (if classifier loaded)
SAMPLING_THRESHOLD      = 500    # subsample landmarks pool above this count
CONVERGENCE_RATE        = 0.005  # stop when new_matches < rate × n_czstack_cells

# Minimum accepted matches to consider the run successful
MIN_MATCH_RATE = 0.50            # require ≥ 50% of CZ cells matched

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CODE_DIR = Path("/root/capsule/code")
DATA_DIR = Path("/root/capsule/data")
SCRATCH  = Path("/scratch")

sys.path.insert(0, str(CODE_DIR))

from coreg_data_loading import (
    load_czstack_centroids,
    load_hcr_centroids,
    load_hcr_scales,
    load_spot_counts,
)
from coreg_matching import run_auto_matching, CZ_RESOLUTION_UM

# Optional: load classifier from step_0 Part B if available
import pickle
classifier = None
classifier_path = DATA_DIR / "match_classifier.pkl"
if classifier_path.exists():
    with open(classifier_path, "rb") as f:
        classifier = pickle.load(f)
    print(f"Classifier loaded from {classifier_path}")
else:
    print("No classifier found — running in MNN-only mode (distance gate).")

In [ ]:
# ── Locate input files ───────────────────────────────────────────────────────
init_dir = SCRATCH / f"{SUBJECT_ID}_{CZSTACK_DATE}_coreg_initial"
if not init_dir.exists():
    raise FileNotFoundError(
        f"Initial alignment directory not found: {init_dir}\n"
        "Run step_2_initial_alignment.ipynb first."
    )

seed_csv = init_dir / f"{SUBJECT_ID}_{CZSTACK_DATE}_landmarks_initial.csv"
rigid_json = init_dir / f"{SUBJECT_ID}_{CZSTACK_DATE}_rigid_initial.json"

if not seed_csv.exists():
    raise FileNotFoundError(f"Seed landmarks not found: {seed_csv}")

seed_df = pd.read_csv(seed_csv)
print(f"Seed landmarks: {len(seed_df)} ({seed_df['active'].sum()} active)")

if rigid_json.exists():
    with open(rigid_json) as f:
        rigid_info = json.load(f)
    print(f"Rigid transform: t={rigid_info['t']}  XY-MNN={rigid_info['xy_mnn_score']}")

# Locate coreg dir + load filepaths
coreg_dirs = sorted(DATA_DIR.glob(f"{SUBJECT_ID}_{CZSTACK_DATE}_ctl-czstack-hcr-coreg_*"))
if not coreg_dirs:
    raise FileNotFoundError(f"No coreg dir for {SUBJECT_ID}/{CZSTACK_DATE}")
coreg_dir = coreg_dirs[-1]
print(f"Coreg dir: {coreg_dir.name}")

fp_iter = sorted(coreg_dir.glob(f"{SUBJECT_ID}_{CZSTACK_DATE}_filepaths_iter.json"))
fp_base = sorted(coreg_dir.glob(f"{SUBJECT_ID}_{CZSTACK_DATE}_filepaths.json"))
with open((fp_iter or fp_base)[0]) as f:
    filepaths = json.load(f)

In [ ]:
# ── Load CZ centroids ────────────────────────────────────────────────────────
czstack_centroid_path = Path(filepaths["czstack_centroid_path"])
if not czstack_centroid_path.exists():
    czstack_centroid_path = coreg_dir / f"{SUBJECT_ID}_{CZSTACK_DATE}_czstack_cell_centroids.csv"
czstack_df = load_czstack_centroids(czstack_centroid_path)
print(f"CZ centroids: {len(czstack_df)} cells")

In [ ]:
# ── Load HCR centroids + spot counts ────────────────────────────────────────
hcr_scales = load_hcr_scales(filepaths["fused_json_file"])
sc = np.array([hcr_scales["scale_z"], hcr_scales["scale_y"], hcr_scales["scale_x"]])

hcr_full_df = load_hcr_centroids(filepaths["hcr_centroid_path"])
print(f"HCR all cells: {len(hcr_full_df)}")

hcr_processed_dir = Path(filepaths["hcr_centroid_path"]).parent.parent
spot_counts = load_spot_counts(
    hcr_processed_dir=hcr_processed_dir,
    hcr_metrics_df=hcr_full_df,
    fallback_coreg_dir=coreg_dir,
    subject_id=SUBJECT_ID,
    czstack_date=CZSTACK_DATE,
)
print(f"Spot counts: {len(spot_counts)} cells, cols={list(spot_counts.columns)}")

# Add is_gfp to hcr_full_df for pool filtering in run_auto_matching
hcr_full_df = hcr_full_df.merge(
    spot_counts[["hcr_cell_id", "counts", "density", "is_gfp"]],
    on="hcr_cell_id", how="left"
).fillna({"counts": 0, "density": 0.0, "is_gfp": False})
hcr_full_df["is_gfp"] = hcr_full_df["is_gfp"].astype(bool)

n_gfp = hcr_full_df["is_gfp"].sum()
print(f"GFP+ cells: {n_gfp} / {len(hcr_full_df)}")

In [ ]:
# ── Load HCR segmentation metrics (volume; optional) ─────────────────────────
hcr_metrics_path = filepaths.get("hcr_segmentation_metrics_path")
if hcr_metrics_path and Path(hcr_metrics_path).exists():
    import pickle as pkl
    with open(hcr_metrics_path, "rb") as f:
        raw_metrics = pkl.load(f)
    # Expect a dict or DataFrame; normalise to DataFrame
    if isinstance(raw_metrics, dict):
        hcr_metrics_df = pd.DataFrame(raw_metrics)
    else:
        hcr_metrics_df = raw_metrics
    # Normalise column names
    rename = {c: "hcr_cell_id" for c in hcr_metrics_df.columns if "id" in c.lower() and "hcr" not in c.lower()}
    hcr_metrics_df = hcr_metrics_df.rename(columns=rename)
    if "hcr_cell_id" not in hcr_metrics_df.columns:
        hcr_metrics_df["hcr_cell_id"] = np.arange(1, len(hcr_metrics_df) + 1)
    print(f"HCR metrics loaded: {len(hcr_metrics_df)} cells, cols={list(hcr_metrics_df.columns)[:6]}")
else:
    hcr_metrics_df = pd.DataFrame(columns=["hcr_cell_id", "volume"])
    print("HCR metrics not found — volume features will be NaN.")

In [ ]:
# ── Run iterative auto-matching ───────────────────────────────────────────────
params = {
    "max_iterations":             MAX_ITERATIONS,
    "k_neighbors":                K_NEIGHBORS,
    "hcr_value_feature":          HCR_VALUE_FEATURE,
    "distance_threshold_um":      DISTANCE_THRESHOLD_UM,
    "accept_probability_threshold": ACCEPT_PROB_THRESHOLD,
    "sampling_threshold":         SAMPLING_THRESHOLD,
    "convergence_rate":           CONVERGENCE_RATE,
}

matches_df = run_auto_matching(
    seed_landmarks=seed_df,
    czstack_df=czstack_df,
    hcr_df=hcr_full_df,
    spot_counts=spot_counts,
    hcr_metrics=hcr_metrics_df,
    hcr_scales=hcr_scales,
    czstack_vol=None,        # no image NCC in step_3 (requires zarr; add in step_0 Part B)
    hcr_zarr_path=None,
    classifier=classifier,   # None → MNN + distance gate
    czstack_res_um=CZ_RESOLUTION_UM,
    params=params,
)

n_matched = len(matches_df)
n_cz = len(czstack_df)
match_rate = n_matched / n_cz
print(f"\nFinal: {n_matched}/{n_cz} CZ cells matched ({100*match_rate:.1f}%)")

assert match_rate >= MIN_MATCH_RATE, (
    f"Match rate {100*match_rate:.1f}% < {100*MIN_MATCH_RATE:.0f}% minimum. "
    "Check seed quality or increase DISTANCE_THRESHOLD_UM."
)

In [ ]:
# ── Diagnostics ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"{SUBJECT_ID} — Auto-matching summary  {n_matched}/{n_cz} ({100*match_rate:.0f}%)")

# Panel 1: matches per iteration
ax = axes[0]
if "iter_matched" in matches_df.columns:
    iter_counts = matches_df.groupby("iter_matched").size()
    ax.bar(iter_counts.index, iter_counts.values, color="steelblue")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("New matches")
    ax.set_title("Matches per iteration")
    # Cumulative line
    ax2 = ax.twinx()
    cumulative = iter_counts.cumsum()
    ax2.plot(cumulative.index, cumulative.values, "r-o", ms=5)
    ax2.set_ylabel("Cumulative matches", color="red")

# Panel 2: match probability histogram (if available)
ax = axes[1]
probs = matches_df["match_probability"].dropna()
if len(probs) > 0:
    ax.hist(probs, bins=40, color="tomato", edgecolor="white")
    ax.axvline(ACCEPT_PROB_THRESHOLD, color="navy", ls="--", label=f"threshold={ACCEPT_PROB_THRESHOLD}")
    ax.set_xlabel("Match probability")
    ax.set_ylabel("Count")
    ax.set_title("Classifier scores")
    ax.legend()
else:
    ax.text(0.5, 0.5, "MNN-only mode\n(no classifier)",
            ha="center", va="center", transform=ax.transAxes, fontsize=14)
    ax.set_title("Classifier scores")

plt.tight_layout()
fig_path = init_dir / "fig_matching_summary.png"
fig.savefig(fig_path, dpi=150)
print(f"Figure saved: {fig_path}")
plt.show()

In [ ]:
# ── Save matches ──────────────────────────────────────────────────────────────
out_csv = init_dir / f"{SUBJECT_ID}_{CZSTACK_DATE}_matches.csv"
matches_df.to_csv(out_csv, index=False)
print(f"Saved {len(matches_df)} matches → {out_csv}")
print(matches_df.head(10).to_string(index=False))